<a href="https://colab.research.google.com/github/Bryan-Kweku-Wilson/Machine-Learning-Projects/blob/main/Spatio_Temporal_GNN_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import networkx as nx

# Load network topology
network_df = pd.read_csv("ecocell_network.csv")

# Load traffic and energy/QoS data
traffic_df = pd.read_csv("ecocell_energy_qos.csv")

# Convert timestamp
traffic_df["timestamp"] = pd.to_datetime(
    traffic_df["timestamp"]
)

# Sort traffic chronologically
traffic_df = traffic_df.sort_values(
    ["timestamp", "bs_id"]
).reset_index(drop=True)

print("Network shape:", network_df.shape)
print("Traffic shape:", traffic_df.shape)

Network shape: (50, 7)
Traffic shape: (36000, 22)


In [2]:
print("\nNetwork columns:")
print(network_df.columns.tolist())

print("\nTraffic columns:")
print(traffic_df.columns.tolist())


Network columns:
['bs_id', 'x', 'y', 'cell_type', 'capacity_mbps', 'coverage_radius_km', 'baseline_power_w']

Traffic columns:
['timestamp', 'bs_id', 'cell_type', 'traffic_mbps', 'active_users', 'capacity_mbps', 'utilization', 'throughput_mbps', 'packet_loss_rate', 'latency_ms', 'traffic_spike', 'idle_power_w', 'baseline_power_w', 'active_power_w', 'light_sleep_power_w', 'deep_sleep_power_w', 'bs_active_energy_kwh', 'light_sleep_energy_kwh', 'deep_sleep_energy_kwh', 'light_sleep_savings_kwh', 'deep_sleep_savings_kwh', 'network_active_energy_kwh']


In [3]:
print("\nNumber of base stations:",
      traffic_df["bs_id"].nunique())

print("Number of timestamps:",
      traffic_df["timestamp"].nunique())

print("Number of traffic observations:",
      len(traffic_df))


Number of base stations: 50
Number of timestamps: 720
Number of traffic observations: 36000


In [4]:
# Create the network graph

G = nx.Graph()

# Add each base station as a node
for _, row in network_df.iterrows():
    G.add_node(
        row["bs_id"],
        x=row["x"],
        y=row["y"],
        cell_type=row["cell_type"],
        capacity_mbps=row["capacity_mbps"],
        coverage_radius_km=row["coverage_radius_km"],
        baseline_power_w=row["baseline_power_w"]
    )

# Create horizontal and vertical connections
for _, row in network_df.iterrows():

    bs_id = row["bs_id"]
    x = row["x"]
    y = row["y"]

    # Connect to the cell on the right
    right_neighbor = network_df[
        (network_df["x"] == x + 1) &
        (network_df["y"] == y)
    ]

    if not right_neighbor.empty:
        neighbor_id = right_neighbor.iloc[0]["bs_id"]

        distance = np.sqrt(
            (x - (x + 1))**2 +
            (y - y)**2
        )

        G.add_edge(
            bs_id,
            neighbor_id,
            distance=distance
        )

    # Connect to the cell above
    upper_neighbor = network_df[
        (network_df["x"] == x) &
        (network_df["y"] == y + 1)
    ]

    if not upper_neighbor.empty:
        neighbor_id = upper_neighbor.iloc[0]["bs_id"]

        distance = np.sqrt(
            (x - x)**2 +
            (y - (y + 1))**2
        )

        G.add_edge(
            bs_id,
            neighbor_id,
            distance=distance
        )

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Number of nodes: 50
Number of edges: 85


In [5]:
node_features = [
    "traffic_mbps",
    "active_users",
    "utilization",
    "throughput_mbps",
    "packet_loss_rate",
    "latency_ms"
]

In [6]:
# Features that will be used by the ST-GNN

node_features = [
    "traffic_mbps",
    "active_users",
    "utilization",
    "throughput_mbps",
    "packet_loss_rate",
    "latency_ms"
]

print("ST-GNN node features:")
for feature in node_features:
    print("-", feature)

ST-GNN node features:
- traffic_mbps
- active_users
- utilization
- throughput_mbps
- packet_loss_rate
- latency_ms


In [7]:
print("\nMissing values:")
print(traffic_df[node_features].isnull().sum())


Missing values:
traffic_mbps        0
active_users        0
utilization         0
throughput_mbps     0
packet_loss_rate    0
latency_ms          0
dtype: int64


In [8]:
# Select the first timestamp
first_timestamp = traffic_df["timestamp"].min()

snapshot_df = traffic_df[
    traffic_df["timestamp"] == first_timestamp
].copy()

# Sort by base-station ID
snapshot_df = snapshot_df.sort_values("bs_id").reset_index(drop=True)

print("Timestamp:", first_timestamp)
print("Snapshot shape:", snapshot_df.shape)

Timestamp: 2026-01-01 00:00:00
Snapshot shape: (50, 22)


In [9]:
X_snapshot = snapshot_df[node_features].values

print("Node feature matrix shape:", X_snapshot.shape)

Node feature matrix shape: (50, 6)


In [10]:
print(snapshot_df[["bs_id"] + node_features].head(10))

  bs_id  traffic_mbps  active_users  utilization  throughput_mbps  \
0  BS00     34.394687            17     0.034926        34.394687   
1  BS01     29.212211            15     0.032913        29.212211   
2  BS02     38.916347            19     0.040130        38.916347   
3  BS03     33.128630            17     0.034967        33.128630   
4  BS04     18.615690             9     0.023300        18.615690   
5  BS05     19.895943            10     0.020705        19.895943   
6  BS06     15.429980             8     0.028351        15.429980   
7  BS07     29.053702            15     0.048585        29.053702   
8  BS08     34.980468            17     0.066934        34.980468   
9  BS09     32.917718            16     0.049675        32.917718   

   packet_loss_rate  latency_ms  
0          0.003310   18.827108  
1          0.009014   20.964499  
2          0.002017   18.629819  
3          0.001163   23.756477  
4          0.002295   21.127977  
5          0.002473   23.547017  
6 

In [11]:
# Sort by timestamp first, then base station
traffic_df = traffic_df.sort_values(
    ["timestamp", "bs_id"]
).reset_index(drop=True)

# Check the first 10 rows
print(
    traffic_df[
        ["timestamp", "bs_id"] + node_features
    ].head(10)
)

   timestamp bs_id  traffic_mbps  active_users  utilization  throughput_mbps  \
0 2026-01-01  BS00     34.394687            17     0.034926        34.394687   
1 2026-01-01  BS01     29.212211            15     0.032913        29.212211   
2 2026-01-01  BS02     38.916347            19     0.040130        38.916347   
3 2026-01-01  BS03     33.128630            17     0.034967        33.128630   
4 2026-01-01  BS04     18.615690             9     0.023300        18.615690   
5 2026-01-01  BS05     19.895943            10     0.020705        19.895943   
6 2026-01-01  BS06     15.429980             8     0.028351        15.429980   
7 2026-01-01  BS07     29.053702            15     0.048585        29.053702   
8 2026-01-01  BS08     34.980468            17     0.066934        34.980468   
9 2026-01-01  BS09     32.917718            16     0.049675        32.917718   

   packet_loss_rate  latency_ms  
0          0.003310   18.827108  
1          0.009014   20.964499  
2          0.0020

In [12]:
X_all = []

timestamps = sorted(
    traffic_df["timestamp"].unique()
)

for timestamp in timestamps:

    snapshot = traffic_df[
        traffic_df["timestamp"] == timestamp
    ].sort_values("bs_id")

    features = snapshot[node_features].values

    X_all.append(features)

X_all = np.array(X_all)

print("All network snapshots shape:", X_all.shape)

All network snapshots shape: (720, 50, 6)


In [13]:
Y_all = []

for timestamp in timestamps:

    snapshot = traffic_df[
        traffic_df["timestamp"] == timestamp
    ].sort_values("bs_id")

    target = snapshot["traffic_mbps"].values

    Y_all.append(target)

Y_all = np.array(Y_all)

print("Target shape:", Y_all.shape)

Target shape: (720, 50)


In [14]:
SEQUENCE_LENGTH = 24

X_sequences = []
Y_targets = []

for i in range(SEQUENCE_LENGTH, len(X_all)):

    # Previous 24 hours
    X_sequences.append(
        X_all[i-SEQUENCE_LENGTH:i]
    )

    # Traffic at the next hour
    Y_targets.append(
        Y_all[i]
    )

X_sequences = np.array(X_sequences)
Y_targets = np.array(Y_targets)

print("X_sequences shape:", X_sequences.shape)
print("Y_targets shape:", Y_targets.shape)

X_sequences shape: (696, 24, 50, 6)
Y_targets shape: (696, 50)


In [15]:
print("First input covers:")
print(timestamps[0])
print("to")
print(timestamps[23])

print("\nFirst target:")
print(timestamps[24])

First input covers:
2026-01-01 00:00:00
to
2026-01-01 23:00:00

First target:
2026-01-02 00:00:00


In [16]:
print("First input sample shape:",
      X_sequences[0].shape)

print("First target sample shape:",
      Y_targets[0].shape)

First input sample shape: (24, 50, 6)
First target sample shape: (50,)


In [17]:
split_index = int(len(X_sequences) * 0.80)

X_train = X_sequences[:split_index]
X_test = X_sequences[split_index:]

Y_train = Y_targets[:split_index]
Y_test = Y_targets[split_index:]

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nX_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

print("\nX_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)

Training samples: 556
Testing samples: 140

X_train shape: (556, 24, 50, 6)
Y_train shape: (556, 50)

X_test shape: (140, 24, 50, 6)
Y_test shape: (140, 50)


In [18]:
from sklearn.preprocessing import StandardScaler

# Reshape training data so each row represents
# one node observation with 6 features

X_train_2d = X_train.reshape(-1, len(node_features))

print("X_train 2D shape:", X_train_2d.shape)

X_train 2D shape: (667200, 6)


In [19]:
scaler = StandardScaler()

X_train_scaled_2d = scaler.fit_transform(
    X_train_2d
)

print("Scaled training shape:",
      X_train_scaled_2d.shape)

Scaled training shape: (667200, 6)


In [20]:
X_train_scaled = X_train_scaled_2d.reshape(
    X_train.shape
)

print("X_train scaled shape:",
      X_train_scaled.shape)

X_train scaled shape: (556, 24, 50, 6)


In [21]:
X_test_2d = X_test.reshape(
    -1,
    len(node_features)
)

X_test_scaled_2d = scaler.transform(
    X_test_2d
)

X_test_scaled = X_test_scaled_2d.reshape(
    X_test.shape
)

print("X_test scaled shape:",
      X_test_scaled.shape)

X_test scaled shape: (140, 24, 50, 6)


In [22]:
print("Training feature means:")
print(X_train_scaled.mean(axis=(0, 1, 2)))

print("\nTraining feature standard deviations:")
print(X_train_scaled.std(axis=(0, 1, 2)))

Training feature means:
[ 2.21186811e-14  1.05618551e-15  8.53894590e-14  2.21186811e-14
 -1.09817577e-13  7.22206855e-13]

Training feature standard deviations:
[1. 1. 1. 1. 1. 1.]


In [23]:
from sklearn.preprocessing import StandardScaler

X_train_2d = X_train.reshape(
    -1,
    len(node_features)
)

scaler = StandardScaler()

X_train_scaled_2d = scaler.fit_transform(
    X_train_2d
)

X_train_scaled = X_train_scaled_2d.reshape(
    X_train.shape
)

In [24]:
X_test_2d = X_test.reshape(
    -1,
    len(node_features)
)

X_test_scaled_2d = scaler.transform(
    X_test_2d
)

X_test_scaled = X_test_scaled_2d.reshape(
    X_test.shape
)

In [25]:
print("Training feature means:")
print(X_train_scaled.mean(axis=(0, 1, 2)))

print("\nTraining feature standard deviations:")
print(X_train_scaled.std(axis=(0, 1, 2)))

Training feature means:
[ 2.21186811e-14  1.05618551e-15  8.53894590e-14  2.21186811e-14
 -1.09817577e-13  7.22206855e-13]

Training feature standard deviations:
[1. 1. 1. 1. 1. 1.]


In [26]:
# Create a consistent mapping from base-station IDs to integer indices

node_ids = sorted(network_df["bs_id"].unique())

node_to_idx = {
    bs_id: idx
    for idx, bs_id in enumerate(node_ids)
}

print("Number of nodes:", len(node_to_idx))
print("\nFirst 10 node mappings:")

for bs_id, idx in list(node_to_idx.items())[:10]:
    print(bs_id, "->", idx)

Number of nodes: 50

First 10 node mappings:
BS00 -> 0
BS01 -> 1
BS02 -> 2
BS03 -> 3
BS04 -> 4
BS05 -> 5
BS06 -> 6
BS07 -> 7
BS08 -> 8
BS09 -> 9


In [27]:
edge_list = []

for source, target in G.edges():

    source_idx = node_to_idx[source]
    target_idx = node_to_idx[target]

    # Add both directions
    edge_list.append([source_idx, target_idx])
    edge_list.append([target_idx, source_idx])

print("Number of directed edges:", len(edge_list))

Number of directed edges: 170


In [28]:
import torch

In [29]:
edge_index = torch.tensor(
    edge_list,
    dtype=torch.long
).t().contiguous()

print("edge_index shape:", edge_index.shape)

edge_index shape: torch.Size([2, 170])


In [30]:
print("First 10 edges:\n")
print(edge_index[:, :10])

First 10 edges:

tensor([[ 0,  1,  0, 10,  1,  2,  1, 11,  2,  3],
        [ 1,  0, 10,  0,  2,  1, 11,  1,  3,  2]])


In [31]:
print("Number of nodes:", len(node_to_idx))
print("Original undirected edges:", G.number_of_edges())
print("Directed edges:", edge_index.shape[1])

print(
    "Minimum node index:",
    edge_index.min().item()
)

print(
    "Maximum node index:",
    edge_index.max().item()
)

Number of nodes: 50
Original undirected edges: 85
Directed edges: 170
Minimum node index: 0
Maximum node index: 49


In [32]:
!pip install -q torch-geometric

import torch
from torch_geometric.data import Data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00


In [33]:
# First training sample
# First hour within that sample

x_sample = torch.tensor(
    X_train_scaled[0, 0],
    dtype=torch.float
)

print("Node feature tensor shape:", x_sample.shape)

Node feature tensor shape: torch.Size([50, 6])


In [34]:
graph_sample = Data(
    x=x_sample,
    edge_index=edge_index
)

print(graph_sample)

Data(x=[50, 6], edge_index=[2, 170])


In [35]:
print("BS00 features:")
print(graph_sample.x[0])

BS00 features:
tensor([-0.8826, -0.8906, -1.0238, -0.8826, -0.8485, -0.6513])


In [36]:
bs00_idx = node_to_idx["BS00"]

neighbor_indices = edge_index[1][
    edge_index[0] == bs00_idx
]

print("BS00 index:", bs00_idx)
print("Neighbor indices:", neighbor_indices)

BS00 index: 0
Neighbor indices: tensor([ 1, 10])


In [37]:
neighbor_ids = [
    node_ids[idx]
    for idx in neighbor_indices.tolist()
]

print("BS00 neighbors:", neighbor_ids)

BS00 neighbors: ['BS01', 'BS10']


In [38]:
# Build the 24 hourly graphs for the first training sample

graph_sequence = []

for t in range(SEQUENCE_LENGTH):

    x_t = torch.tensor(
        X_train_scaled[0, t],
        dtype=torch.float
    )

    graph_t = Data(
        x=x_t,
        edge_index=edge_index
    )

    graph_sequence.append(graph_t)

print("Number of graphs:", len(graph_sequence))

Number of graphs: 24


In [39]:
for t, graph in enumerate(graph_sequence[:3]):

    print(
        f"Hour {t}:",
        graph
    )

Hour 0: Data(x=[50, 6], edge_index=[2, 170])
Hour 1: Data(x=[50, 6], edge_index=[2, 170])
Hour 2: Data(x=[50, 6], edge_index=[2, 170])


In [40]:
same_edges = all(
    torch.equal(
        graph_sequence[0].edge_index,
        graph.edge_index
    )
    for graph in graph_sequence
)

print("Same topology across all 24 hours:", same_edges)

Same topology across all 24 hours: True


In [41]:
print("BS00 at hour 0:")
print(graph_sequence[0].x[0])

print("\nBS00 at hour 1:")
print(graph_sequence[1].x[0])

BS00 at hour 0:
tensor([-0.8826, -0.8906, -1.0238, -0.8826, -0.8485, -0.6513])

BS00 at hour 1:
tensor([-1.0176, -1.0138, -1.1187, -1.0176, -1.3014, -0.1011])


In [42]:
target_first_sample = Y_train[0]

print(
    "Target shape:",
    target_first_sample.shape
)

print(
    "\nFirst 10 target traffic values:"
)

print(target_first_sample[:10])

Target shape: (50,)

First 10 target traffic values:
[31.49107936 28.347815   37.45189485 31.19055726 20.01631515 19.0445568
 16.06403015 25.13813005 34.68133231 33.55278806]


In [43]:
# Build the 24 hourly graphs for the first training sample

graph_sequence = []

for t in range(SEQUENCE_LENGTH):

    x_t = torch.tensor(
        X_train_scaled[0, t],
        dtype=torch.float
    )

    graph_t = Data(
        x=x_t,
        edge_index=edge_index
    )

    graph_sequence.append(graph_t)

print("Number of graphs:", len(graph_sequence))

Number of graphs: 24


In [44]:
for t, graph in enumerate(graph_sequence[:3]):
    print(f"Hour {t}:", graph)

Hour 0: Data(x=[50, 6], edge_index=[2, 170])
Hour 1: Data(x=[50, 6], edge_index=[2, 170])
Hour 2: Data(x=[50, 6], edge_index=[2, 170])


In [45]:
same_edges = all(
    torch.equal(
        graph_sequence[0].edge_index,
        graph.edge_index
    )
    for graph in graph_sequence
)

print("Same topology across all 24 hours:", same_edges)

Same topology across all 24 hours: True


In [46]:
print("BS00 at hour 0:")
print(graph_sequence[0].x[0])

print("\nBS00 at hour 1:")
print(graph_sequence[1].x[0])

BS00 at hour 0:
tensor([-0.8826, -0.8906, -1.0238, -0.8826, -0.8485, -0.6513])

BS00 at hour 1:
tensor([-1.0176, -1.0138, -1.1187, -1.0176, -1.3014, -0.1011])


In [47]:
import torch
import torch.nn as nn

from torch_geometric.nn import GCNConv

In [48]:
class STGNN(nn.Module):

    def __init__(
        self,
        input_features=6,
        hidden_dim=32,
        output_dim=1,
        dropout=0.2
    ):
        super().__init__()

        # Spatial layers
        self.gcn1 = GCNConv(
            input_features,
            hidden_dim
        )

        self.gcn2 = GCNConv(
            hidden_dim,
            hidden_dim
        )

        # Temporal layer
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Prediction layer
        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x_sequence, edge_index):

        spatial_outputs = []

        # Process each hour separately through the GNN
        for t in range(x_sequence.shape[1]):

            x_t = x_sequence[:, t, :, :]

            # x_t shape:
            # (batch, nodes, features)

            batch_outputs = []

            for b in range(x_t.shape[0]):

                node_features = x_t[b]

                h = self.gcn1(
                    node_features,
                    edge_index
                )

                h = torch.relu(h)

                h = self.dropout(h)

                h = self.gcn2(
                    h,
                    edge_index
                )

                h = torch.relu(h)

                batch_outputs.append(h)

            h_t = torch.stack(batch_outputs)

            spatial_outputs.append(h_t)

        # Shape:
        # (batch, time, nodes, hidden_dim)
        spatial_outputs = torch.stack(
            spatial_outputs,
            dim=1
        )

        # Rearrange so each base station has
        # its own temporal sequence
        batch_size = spatial_outputs.shape[0]
        time_steps = spatial_outputs.shape[1]
        num_nodes = spatial_outputs.shape[2]
        hidden_dim = spatial_outputs.shape[3]

        gru_input = spatial_outputs.permute(
            0, 2, 1, 3
        )

        # (batch, nodes, time, hidden_dim)

        gru_input = gru_input.reshape(
            batch_size * num_nodes,
            time_steps,
            hidden_dim
        )

        # GRU
        gru_output, _ = self.gru(gru_input)

        # Take the final time step
        final_hidden = gru_output[:, -1, :]

        # Predict next-hour traffic
        predictions = self.fc(final_hidden)

        # Restore batch/node structure
        predictions = predictions.reshape(
            batch_size,
            num_nodes
        )

        return predictions

In [49]:
stgnn_model = STGNN(
    input_features=6,
    hidden_dim=32,
    output_dim=1,
    dropout=0.2
)

In [50]:
print(stgnn_model)

STGNN(
  (gcn1): GCNConv(6, 32)
  (gcn2): GCNConv(32, 32)
  (gru): GRU(32, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [51]:
total_params = sum(
    p.numel()
    for p in stgnn_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", total_params)

Trainable parameters: 7649


In [52]:
X_train_tensor = torch.tensor(
    X_train_scaled[:2],
    dtype=torch.float
)

Y_train_tensor = torch.tensor(
    Y_train[:2],
    dtype=torch.float
)

print("Input shape:", X_train_tensor.shape)
print("Target shape:", Y_train_tensor.shape)

Input shape: torch.Size([2, 24, 50, 6])
Target shape: torch.Size([2, 50])


In [53]:
with torch.no_grad():

    test_output = stgnn_model(
        X_train_tensor,
        edge_index
    )

print("Model output shape:", test_output.shape)

Model output shape: torch.Size([2, 50])


In [54]:
# Chronological train/validation split

val_split_index = int(len(X_train) * 0.80)

X_train_stgnn = X_train_scaled[:val_split_index]
Y_train_stgnn = Y_train[:val_split_index]

X_val_stgnn = X_train_scaled[val_split_index:]
Y_val_stgnn = Y_train[val_split_index:]

print("ST-GNN training input:", X_train_stgnn.shape)
print("ST-GNN training target:", Y_train_stgnn.shape)

print("ST-GNN validation input:", X_val_stgnn.shape)
print("ST-GNN validation target:", Y_val_stgnn.shape)

print("ST-GNN test input:", X_test_scaled.shape)
print("ST-GNN test target:", Y_test.shape)

ST-GNN training input: (444, 24, 50, 6)
ST-GNN training target: (444, 50)
ST-GNN validation input: (112, 24, 50, 6)
ST-GNN validation target: (112, 50)
ST-GNN test input: (140, 24, 50, 6)
ST-GNN test target: (140, 50)


In [55]:
from sklearn.preprocessing import StandardScaler

target_scalers = {}

Y_train_scaled = np.zeros_like(Y_train_stgnn, dtype=float)
Y_val_scaled = np.zeros_like(Y_val_stgnn, dtype=float)
Y_test_scaled = np.zeros_like(Y_test, dtype=float)

for node_idx in range(Y_train_stgnn.shape[1]):

    scaler_target = StandardScaler()

    # Fit ONLY on training targets
    train_values = Y_train_stgnn[:, node_idx].reshape(-1, 1)

    scaler_target.fit(train_values)

    # Transform train, validation and test
    Y_train_scaled[:, node_idx] = (
        scaler_target
        .transform(
            Y_train_stgnn[:, node_idx].reshape(-1, 1)
        )
        .ravel()
    )

    Y_val_scaled[:, node_idx] = (
        scaler_target
        .transform(
            Y_val_stgnn[:, node_idx].reshape(-1, 1)
        )
        .ravel()
    )

    Y_test_scaled[:, node_idx] = (
        scaler_target
        .transform(
            Y_test[:, node_idx].reshape(-1, 1)
        )
        .ravel()
    )

    target_scalers[node_idx] = scaler_target

In [56]:
print("Scaled training target:", Y_train_scaled.shape)
print("Scaled validation target:", Y_val_scaled.shape)
print("Scaled test target:", Y_test_scaled.shape)

Scaled training target: (444, 50)
Scaled validation target: (112, 50)
Scaled test target: (140, 50)


In [57]:
X_train_tensor = torch.tensor(
    X_train_stgnn,
    dtype=torch.float32
)

Y_train_tensor = torch.tensor(
    Y_train_scaled,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_stgnn,
    dtype=torch.float32
)

Y_val_tensor = torch.tensor(
    Y_val_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

Y_test_tensor = torch.tensor(
    Y_test_scaled,
    dtype=torch.float32
)

In [58]:
print("X_train:", X_train_tensor.shape)
print("Y_train:", Y_train_tensor.shape)

print("X_val:", X_val_tensor.shape)
print("Y_val:", Y_val_tensor.shape)

print("X_test:", X_test_tensor.shape)
print("Y_test:", Y_test_tensor.shape)

X_train: torch.Size([444, 24, 50, 6])
Y_train: torch.Size([444, 50])
X_val: torch.Size([112, 24, 50, 6])
Y_val: torch.Size([112, 50])
X_test: torch.Size([140, 24, 50, 6])
Y_test: torch.Size([140, 50])


In [59]:
optimizer = torch.optim.Adam(
    stgnn_model.parameters(),
    lr=0.001
)

criterion = nn.MSELoss()

In [60]:
def train_one_epoch(
    model,
    X,
    Y,
    edge_index,
    optimizer,
    criterion,
    batch_size=8
):

    model.train()

    total_loss = 0.0
    num_samples = X.shape[0]

    for start in range(0, num_samples, batch_size):

        end = start + batch_size

        X_batch = X[start:end]
        Y_batch = Y[start:end]

        optimizer.zero_grad()

        predictions = model(
            X_batch,
            edge_index
        )

        loss = criterion(
            predictions,
            Y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * X_batch.shape[0]

    average_loss = total_loss / num_samples

    return average_loss

In [61]:
def evaluate_loss(
    model,
    X,
    Y,
    edge_index,
    criterion,
    batch_size=8
):

    model.eval()

    total_loss = 0.0
    num_samples = X.shape[0]

    with torch.no_grad():

        for start in range(0, num_samples, batch_size):

            end = start + batch_size

            X_batch = X[start:end]
            Y_batch = Y[start:end]

            predictions = model(
                X_batch,
                edge_index
            )

            loss = criterion(
                predictions,
                Y_batch
            )

            total_loss += loss.item() * X_batch.shape[0]

    average_loss = total_loss / num_samples

    return average_loss

In [62]:
EPOCHS = 30

train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        stgnn_model,
        X_train_tensor,
        Y_train_tensor,
        edge_index,
        optimizer,
        criterion,
        batch_size=8
    )

    val_loss = evaluate_loss(
        stgnn_model,
        X_val_tensor,
        Y_val_tensor,
        edge_index,
        criterion,
        batch_size=8
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f}"
    )

Epoch 01/30 | Train Loss: 0.8028 | Val Loss: 0.4419
Epoch 02/30 | Train Loss: 0.2928 | Val Loss: 0.2248
Epoch 03/30 | Train Loss: 0.2148 | Val Loss: 0.1907
Epoch 04/30 | Train Loss: 0.1960 | Val Loss: 0.1766
Epoch 05/30 | Train Loss: 0.1891 | Val Loss: 0.1721
Epoch 06/30 | Train Loss: 0.1842 | Val Loss: 0.1698
Epoch 07/30 | Train Loss: 0.1821 | Val Loss: 0.1670
Epoch 08/30 | Train Loss: 0.1770 | Val Loss: 0.1628
Epoch 09/30 | Train Loss: 0.1744 | Val Loss: 0.1598
Epoch 10/30 | Train Loss: 0.1703 | Val Loss: 0.1539
Epoch 11/30 | Train Loss: 0.1671 | Val Loss: 0.1506
Epoch 12/30 | Train Loss: 0.1640 | Val Loss: 0.1450
Epoch 13/30 | Train Loss: 0.1587 | Val Loss: 0.1407
Epoch 14/30 | Train Loss: 0.1577 | Val Loss: 0.1381
Epoch 15/30 | Train Loss: 0.1549 | Val Loss: 0.1350
Epoch 16/30 | Train Loss: 0.1520 | Val Loss: 0.1324
Epoch 17/30 | Train Loss: 0.1499 | Val Loss: 0.1315
Epoch 18/30 | Train Loss: 0.1476 | Val Loss: 0.1309
Epoch 19/30 | Train Loss: 0.1457 | Val Loss: 0.1298
Epoch 20/30 

In [63]:
stgnn_model.eval()

STGNN(
  (gcn1): GCNConv(6, 32)
  (gcn2): GCNConv(32, 32)
  (gru): GRU(32, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

In [64]:
with torch.no_grad():

    stgnn_predictions_scaled = stgnn_model(
        X_test_tensor,
        edge_index
    )

print(
    "Prediction shape:",
    stgnn_predictions_scaled.shape
)

Prediction shape: torch.Size([140, 50])


In [65]:
stgnn_predictions_scaled

tensor([[-1.3033, -1.4067, -1.3864,  ..., -1.3759, -1.3888, -1.2807],
        [-1.1781, -1.3177, -1.2865,  ..., -1.3097, -1.3286, -1.1967],
        [-1.0199, -1.1843, -1.1394,  ..., -1.1418, -1.1796, -1.0355],
        ...,
        [ 0.4069,  0.6656,  0.9065,  ...,  0.7813,  0.7388,  0.6646],
        [-0.1068,  0.0084,  0.1568,  ...,  0.1269,  0.4191,  0.3658],
        [-0.6227, -0.5127, -0.3710,  ..., -0.5784, -0.5771, -0.6571]])

In [66]:
stgnn_predictions_mbps = np.zeros_like(
    stgnn_predictions_scaled.numpy()
)

for node_idx in range(50):

    scaler_target = target_scalers[node_idx]

    predictions_node = (
        stgnn_predictions_scaled[:, node_idx]
        .numpy()
        .reshape(-1, 1)
    )

    stgnn_predictions_mbps[:, node_idx] = (
        scaler_target
        .inverse_transform(predictions_node)
        .ravel()
    )

In [67]:
print(
    "Prediction shape:",
    stgnn_predictions_mbps.shape
)

print(
    "First test timestamp predictions:"
)

print(stgnn_predictions_mbps[0])

Prediction shape: (140, 50)
First test timestamp predictions:
[24.825312  15.880979  21.424652  17.23677   10.553059  13.951546
 10.79583   15.140803  20.926474  29.503874   7.578866  13.404091
 11.00451   12.304595   9.01026   11.30801   15.155185  16.290243
 17.417662  17.90991   18.178167   9.052967  16.528448  19.271124
 21.384378  22.656233  13.992235  15.645429  17.918976   8.9659815
 16.649385  10.093658   7.284594  14.628657  16.2514    14.535468
 14.163967   8.195801  18.575294  18.308886  14.24719   14.829631
  9.457251  17.8104    17.905436  23.270134  15.133871  15.623454
 20.729977  16.03155  ]


In [68]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

actual_mbps = Y_test

predicted_mbps = stgnn_predictions_mbps

stgnn_mae = mean_absolute_error(
    actual_mbps.flatten(),
    predicted_mbps.flatten()
)

stgnn_rmse = np.sqrt(
    mean_squared_error(
        actual_mbps.flatten(),
        predicted_mbps.flatten()
    )
)

print("ST-GNN MAE:", stgnn_mae)
print("ST-GNN RMSE:", stgnn_rmse)

ST-GNN MAE: 8.87656350915404
ST-GNN RMSE: 16.9913369616072


In [69]:
comparison = pd.DataFrame({
    "Model": [
        "Naive",
        "Random Forest",
        "GRU",
        "ST-GNN"
    ],
    "MAE": [
        23.899173,
        8.233464,
        7.868627,
        stgnn_mae
    ],
    "RMSE": [
        35.502109,
        16.681637,
        16.096340,
        stgnn_rmse
    ]
})

comparison

,Model,MAE,RMSE
0,Naive,23.899173,35.502109
1,Random Forest,8.233464,16.681637
2,GRU,7.868627,16.096340
3,ST-GNN,8.876564,16.991337


In [70]:
stgnn_results = pd.DataFrame({
    "actual_mbps": actual_mbps.flatten(),
    "predicted_mbps": predicted_mbps.flatten()
})

stgnn_results["absolute_error"] = (
    np.abs(
        stgnn_results["actual_mbps"]
        - stgnn_results["predicted_mbps"]
    )
)

stgnn_results["signed_error"] = (
    stgnn_results["actual_mbps"]
    - stgnn_results["predicted_mbps"]
)

stgnn_results["squared_error"] = (
    stgnn_results["actual_mbps"]
    - stgnn_results["predicted_mbps"]
) ** 2

In [71]:
print(stgnn_results.shape)
print(stgnn_results.head())

(7000, 5)
   actual_mbps  predicted_mbps  absolute_error  signed_error  squared_error
0    19.564063       24.825312        5.261249     -5.261249      27.680742
1    15.887339       15.880979        0.006361      0.006361       0.000040
2    24.898465       21.424652        3.473813      3.473813      12.067376
3    19.332031       17.236771        2.095260      2.095260       4.390115
4    11.088532       10.553059        0.535473      0.535473       0.286731


In [72]:
stgnn_results["traffic_level"] = pd.qcut(
    stgnn_results["actual_mbps"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [73]:
stgnn_traffic_mae = (
    stgnn_results
    .groupby("traffic_level", observed=True)["absolute_error"]
    .mean()
)

print(stgnn_traffic_mae)

traffic_level
Low           3.304455
Medium        7.311356
High          9.156893
Very High    15.733549
Name: absolute_error, dtype: float64


In [74]:
test_spike_flags = []

# Define the test dataframe and base station list using our test split
test_df = traffic_df.iloc[split_index * len(node_ids):].copy()
test_bs_ids = node_ids

for bs_id in test_bs_ids:

    bs_test = test_df[
        test_df["bs_id"] == bs_id
    ].sort_values("timestamp")

    test_spike_flags.extend(
        bs_test["traffic_spike"].values
    )

In [75]:
stgnn_spike_flags = []

test_timestamps = timestamps[
    SEQUENCE_LENGTH + split_index:
]

In [76]:
print("First test timestamp:")
print(test_timestamps[0])

print("\nLast test timestamp:")
print(test_timestamps[-1])

print("\nNumber of test timestamps:")
print(len(test_timestamps))

First test timestamp:
2026-01-25 04:00:00

Last test timestamp:
2026-01-30 23:00:00

Number of test timestamps:
140


In [77]:
stgnn_spike_matrix = []

for timestamp in test_timestamps:

    snapshot = traffic_df[
        traffic_df["timestamp"] == timestamp
    ].sort_values("bs_id")

    spike_values = snapshot["traffic_spike"].values

    stgnn_spike_matrix.append(spike_values)

stgnn_spike_matrix = np.array(stgnn_spike_matrix)

print("Spike matrix shape:", stgnn_spike_matrix.shape)

Spike matrix shape: (140, 50)


In [78]:
stgnn_results["traffic_spike"] = (
    stgnn_spike_matrix.flatten()
)

print(
    stgnn_results["traffic_spike"].value_counts()
)

traffic_spike
False    6862
True      138
Name: count, dtype: int64


In [79]:
stgnn_spike_mae = (
    stgnn_results
    .groupby("traffic_spike")["absolute_error"]
    .mean()
)

stgnn_spike_rmse = (
    stgnn_results
    .groupby("traffic_spike")["squared_error"]
    .mean()
    .apply(np.sqrt)
)

print("ST-GNN MAE by traffic spike:")
print(stgnn_spike_mae)

print("\nST-GNN RMSE by traffic spike:")
print(stgnn_spike_rmse)

ST-GNN MAE by traffic spike:
traffic_spike
False     7.390228
True     82.784040
Name: absolute_error, dtype: float64

ST-GNN RMSE by traffic spike:
traffic_spike
False     9.978534
True     98.454784
Name: squared_error, dtype: float64


In [80]:
spike_errors = stgnn_results[
    stgnn_results["traffic_spike"] == True
]["signed_error"]

print("Mean spike signed error:", spike_errors.mean())
print("Median spike signed error:", spike_errors.median())

print(
    "Spike underpredictions:",
    (spike_errors > 0).sum()
)

print(
    "Spike overpredictions:",
    (spike_errors < 0).sum()
)

Mean spike signed error: 82.7840396496126
Median spike signed error: 73.49248297017814
Spike underpredictions: 138
Spike overpredictions: 0


In [81]:
comparison = pd.DataFrame({
    "Model": [
        "Naive",
        "Random Forest",
        "GRU",
        "ST-GNN"
    ],
    "MAE_Mbps": [
        23.899173,
        8.233464,
        7.868627,
        stgnn_mae
    ],
    "RMSE_Mbps": [
        35.502109,
        16.681637,
        16.096340,
        stgnn_rmse
    ]
})

comparison["MAE_improvement_vs_naive_pct"] = (
    (comparison["MAE_Mbps"].iloc[0] - comparison["MAE_Mbps"])
    / comparison["MAE_Mbps"].iloc[0] * 100
)

comparison.round(3)

,Model,MAE_Mbps,RMSE_Mbps,MAE_improvement_vs_naive_pct
0,Naive,23.899,35.502,0.000
1,Random Forest,8.233,16.682,65.549
2,GRU,7.869,16.096,67.076
3,ST-GNN,8.877,16.991,62.858
